<h1>Phishing website evaluation using decision trees and random forests</h1>

<h2>1. Business understanding</h2>

The pipeline uses data from https://archive.ics.uci.edu/dataset/327/phishing+websites. The dataset contains a set of URLs that have been evaluated using a list of features that have been proved effective in predicting whether or not the URL leads to **1 = legitimate** or **-1 = phishing** website.

Our goal is to create a pipeline that uses decision trees and random forests to reliably predict whether or not any given website leads to a legitimate or phishing website based on the easily obtainable information about the website. If successful, it could be used to create automated system to warn users of dangerous websites.

<h2>2. Data understanding</h2>

In [ ]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import matplotlib.pyplot as plt

# fetch dataset 
df = fetch_ucirepo(id=327) 

# data (as pandas dataframes) 
X = df.data.features  # input features
y = df.data.targets # target features

In [ ]:
# metadata 
print(json.dumps(df.metadata, indent=2))

Metadata above contains indepth information about the dataset, its creators, date and original links to its publication.

In [ ]:
# variable information 
print(df.variables) 

Dataset does not appear to have any missing values and its types are correctly assigned.

In [9]:
X.describe(include='all')

,having_ip_address,url_length,shortining_service,having_at_symbol,double_slash_redirecting,prefix_suffix,having_sub_domain,sslfinal_state,domain_registration_length,favicon,...,rightclick,popupwindow,iframe,age_of_domain,dnsrecord,web_traffic,page_rank,google_index,links_pointing_to_page,statistical_report
count,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,...,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000,11055.000000
mean,0.313795,-0.633198,0.738761,0.700588,0.741474,-0.734962,0.063953,0.250927,-0.336771,0.628584,...,0.913885,0.613388,0.816915,0.061239,0.377114,0.287291,-0.483673,0.721574,0.344007,0.719584
std,0.949534,0.766095,0.673998,0.713598,0.671011,0.678139,0.817518,0.911892,0.941629,0.777777,...,0.405991,0.789818,0.576784,0.998168,0.926209,0.827733,0.875289,0.692369,0.569944,0.694437
min,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
25%,-1.000000,-1.000000,1.000000,1.000000,1.000000,-1.000000,-1.000000,-1.000000,-1.000000,1.000000,...,1.000000,1.000000,1.000000,-1.000000,-1.000000,0.000000,-1.000000,1.000000,0.000000,1.000000
50%,1.000000,-1.000000,1.000000,1.000000,1.000000,-1.000000,0.000000,1.000000,-1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,0.000000,1.000000
75%,1.000000,-1.000000,1.000000,1.000000,1.000000,-1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


Phishing websites dataframe contains 11055 sample URLs with 30 different features, contained in the variable `X`. Each feature has a value of 1 (legitimate), 0 (suspicious) or -1 (phishing). Not every feature can have 0 as value. Results 1 (legitimate) or -1 (phishing) are contained in the variable `y`.

We can observe that the values presented do not differ in magnitude. Nor do they contain any outliers, as both mean and max are in all cases within min and max.

<h2>3. Data preparation</h2>

To prepare the data for modeling, we must first split it to training data and testing data. As we had already observed, in absense of differences in magnitude, scaling of values is not necessary.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state=20)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

<h2>4. Modeling</h2>

 - Valitaan koneoppimismenetelmä, opetetaan malli ja validoidaan se (validation).
 - Dokumentointiin on sisällytettävä tarkasti valittu menetelmä, käytetyt parametrit sekä mitattu mallin suorituskyky.
 - Millä perusteella tehty valinnat
 - Taulukko on hyvä viestimiseen
 - Ei tarvitse olla yksityiskohtainen, mutta selitetään miksi on päädytty lopputulokseen.

We have chosen **decision tree** from **Scikit-learn Python library** as our machine learning method to train the model.

An instance of `DecisionTreeClassifier` class is created and the model is trained using its `fit` method. Parameters `max_depth` is notable for restricting the size of the tree.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

model = DecisionTreeClassifier(max_depth=2, random_state=20)
model.fit(X_train, y_train)

Using the constructed classifier, the tree can now be visualized. Changing the earlier `max_depth` can dramatically alter the complexity of visualization, in which case altering `figsize` is often necessary for readability.

Names of classes must be given as parameters.

In [ ]:
fig = plt.figure(figsize = (8, 8))
plot_tree(model, feature_names = X.keys(), class_names = ['phishing', 'legitimate'])
plt.show()

We can test the performance of our classifier by giving it previously unseen data. `Predict` method determines the sample classes, which are then compared with the true labels. Results are presented with `confusion_matrix`.

In [ ]:
from sklearn.metrics import confusion_matrix

preds = model.predict(X_test)
confusion_matrix(y_test, preds)

<h2>5. Evaluation</h2>

 - Arvioidaan mallin todellinen onnistuminen.
 - Selvitetään, kuinka hyvin malli vastaa sille alun perin asetettuihin liiketoimintavaatimuksiin (business requirements).<br>
 - Ei katsota mallin suorituskykyä (tämä tehdään kappaleessa 4), vaan palataan takaisin kappaleeseen 1. ja millä tavalla saavutetut tulokset vertautuu alkuperäisiin tavoitteisiin.

<h2>6. Deployment</h2>

- Malli viedään käytäntöön (deployment).
- Tähän sisältyy tulosten viestiminen ja suositusten luominen siitä, miten mallia tulisi hyödyntää käytännössä tai mitä jatkotoimenpiteitä tulisi tehdä.